# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 36.5234


In [3]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Fetch Results

In [4]:
import seml
import pandas as pd

db_name = 'daro'
#states=["FAILED"]
states = ["COMPLETED"]

all_results = seml.evaluation.get_results(db_name, to_data_frame=True, states=states)
print(f"Lenght of all_results: {len(all_results)}")
print(all_results.columns)

all_results.head()

/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-package
s/rich/live.py:231: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Lenght of all_results: 144
Index(['_id', 'config.overwrite', 'config.db_collection', 'config.batch_size',
       'config.calib_dataset_name', 'config.calib_dataset_split',
       'config.calib_seq_length', 'config.clean_cache', 'config.device',
       'config.eval_dataset_name', 'config.eval_dataset_split',
       'config.eval_metrics', 'config.eval_n_samples',
       'config.eval_seq_length', 'config.model_name', 'config.quantize_method',
       'config.quantized_model_save_path', 'config.save_quantized_model',
       'config.seed', 'result.current_gpu_type',
       'result.current_gpu_total_memory', 'result.current_gpu_free_memory',
       'result.perplexity', 'result.brier_score', 'result.disk_space_usage',
       'result.quantize_runtime', 'result.fail_trace'],
      dtype='object')


,_id,config.overwrite,config.db_collection,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.clean_cache,config.device,config.eval_dataset_name,...,config.save_quantized_model,config.seed,result.current_gpu_type,result.current_gpu_total_memory,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime,result.fail_trace
0,1,1,daro,1,WikiText,validation,2048,True,cuda,WikiText,...,True,70152725,NVIDIA A100-PCIE-40GB,40339.3125,[23.896484375],5.616476,0.000004,nan,nan,<function get_results at 0x7f4550458c10>
1,2,2,daro,1,WikiText,validation,2048,True,cuda,WikiText,...,True,128949945,NVIDIA A100-PCIE-40GB,40339.3125,[31.123046875],6.307941,0.000004,15.14 GB,15.353248,<function get_results at 0x7f4550458c10>
2,3,3,daro,1,WikiText,validation,2048,True,cuda,WikiText,...,True,487086334,NVIDIA A100-PCIE-40GB,40339.3125,[28.4453125],5.697184,0.000004,10.42 GB,26.787766,<function get_results at 0x7f4550458c10>
3,4,4,daro,1,WikiText,validation,2048,True,cuda,WikiText,...,True,812645686,NVIDIA A100-PCIE-40GB,40339.3125,[35.19140625],8.559265,0.000005,5.33 GB,994.555119,<function get_results at 0x7f4550458c10>
4,5,5,daro,1,WikiText,validation,2048,True,cuda,WikiText,...,True,163493064,NVIDIA A100-PCIE-40GB,40339.3125,[29.294921875],5.615966,0.000004,nan,11.029122,<function get_results at 0x7f4550458c10>


In [3]:
nan_perplexity_df = all_results[all_results['result.perplexity'].isna() & all_results['config.overwrite'].notna()]
print(nan_perplexity_df['config.quantize_method'])

Series([], Name: config.quantize_method, dtype: object)


In [5]:
columns_to_remove = [
    'config.overwrite',
    'config.db_collection',
    'config.clean_cache',
    'config.device',
    'config.quantized_model_save_path',
    'config.save_quantized_model',
    'config.seed',
    'result',
    'result.current_gpu_type',
    'result.current_gpu_total_memory',
    'result.fail_trace'
]

all_results = all_results.drop(columns=columns_to_remove, errors='ignore')
all_results.columns

Index(['_id', 'config.batch_size', 'config.calib_dataset_name',
       'config.calib_dataset_split', 'config.calib_seq_length',
       'config.eval_dataset_name', 'config.eval_dataset_split',
       'config.eval_metrics', 'config.eval_n_samples',
       'config.eval_seq_length', 'config.model_name', 'config.quantize_method',
       'result.current_gpu_free_memory', 'result.perplexity',
       'result.brier_score', 'result.disk_space_usage',
       'result.quantize_runtime'],
      dtype='object')

In [6]:
all_results.head()

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.eval_n_samples,config.eval_seq_length,config.model_name,config.quantize_method,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime
0,1,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,NONE,[23.896484375],5.616476,0.000004,nan,nan
1,2,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,BNB-4,[31.123046875],6.307941,0.000004,15.14 GB,15.353248
2,3,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,BNB-8,[28.4453125],5.697184,0.000004,10.42 GB,26.787766
3,4,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,AWQ-4,[35.19140625],8.559265,0.000005,5.33 GB,994.555119
4,5,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,HQQ-8-uniform,[29.294921875],5.615966,0.000004,nan,11.029122


In [7]:
from src.algorithms.quantization import QUANT_CONFIGS

# Define a function to label datasets explicitly
def label_dataset(row):
    return f"Calib: {row['config.calib_dataset_name']} ({row['config.calib_dataset_split']}) | Eval: {row['config.eval_dataset_name']} ({row['config.eval_dataset_split']})"

# Adding new columns to check if splits and datasets are the same
all_results['split_equals'] = all_results['config.calib_dataset_split'] == all_results['config.eval_dataset_split']
all_results['dataset_equals'] = all_results['config.calib_dataset_name'] == all_results['config.eval_dataset_name']
all_results['dataset_label'] = all_results.apply(label_dataset, axis=1)

# Function to extract quantize_method and n_bits
def extract_quantize_method_and_bits(quantize_method):
    config = QUANT_CONFIGS[quantize_method]
    quantize_method = config['quantize_method']
    n_bits = config.get('num_bits', None)
    return quantize_method, n_bits

# Apply the function to create new columns
all_results[['config.quantize_method_type', 'config.n_bits']] = all_results['config.quantize_method'].apply(
    lambda x: pd.Series(extract_quantize_method_and_bits(x))
)

# Rename the column
all_results.head()

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.eval_n_samples,config.eval_seq_length,...,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime,split_equals,dataset_equals,dataset_label,config.quantize_method_type,config.n_bits
0,1,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[23.896484375],5.616476,0.000004,nan,nan,False,True,Calib: WikiText (validation) | Eval: WikiText ...,NONE,16
1,2,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[31.123046875],6.307941,0.000004,15.14 GB,15.353248,False,True,Calib: WikiText (validation) | Eval: WikiText ...,BNB,4
2,3,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[28.4453125],5.697184,0.000004,10.42 GB,26.787766,False,True,Calib: WikiText (validation) | Eval: WikiText ...,BNB,8
3,4,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[35.19140625],8.559265,0.000005,5.33 GB,994.555119,False,True,Calib: WikiText (validation) | Eval: WikiText ...,AWQ,4
4,5,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[29.294921875],5.615966,0.000004,nan,11.029122,False,True,Calib: WikiText (validation) | Eval: WikiText ...,HQQ,8


In [8]:
all_results.loc[all_results['config.quantize_method_type'] == 'QUANTO']

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.eval_n_samples,config.eval_seq_length,...,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime,split_equals,dataset_equals,dataset_label,config.quantize_method_type,config.n_bits
7,8,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,5.387069,False,True,Calib: WikiText (validation) | Eval: WikiText ...,QUANTO,8
16,20,1,WikiText,validation,2048,OpenAssistant,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,4.634208,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,QUANTO,8
25,32,1,WikiText,validation,2048,C4,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,4.544218,False,False,Calib: WikiText (validation) | Eval: C4 (test),QUANTO,8
34,44,1,WikiText,validation,2048,PTB,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,5.346802,False,False,Calib: WikiText (validation) | Eval: PTB (test),QUANTO,8
43,56,1,OpenAssistant,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,5.698889,False,False,Calib: OpenAssistant (validation) | Eval: Wiki...,QUANTO,8
52,68,1,OpenAssistant,validation,2048,OpenAssistant,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,5.750916,False,True,Calib: OpenAssistant (validation) | Eval: Open...,QUANTO,8
61,80,1,OpenAssistant,validation,2048,C4,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,4.839127,False,False,Calib: OpenAssistant (validation) | Eval: C4 (...,QUANTO,8
70,92,1,OpenAssistant,validation,2048,PTB,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,6.161056,False,False,Calib: OpenAssistant (validation) | Eval: PTB ...,QUANTO,8
79,104,1,C4,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,7.37031,False,False,Calib: C4 (validation) | Eval: WikiText (test),QUANTO,8
88,116,1,C4,validation,2048,OpenAssistant,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[18.265625],inf,0.000008,nan,4.780027,False,False,Calib: C4 (validation) | Eval: OpenAssistant (...,QUANTO,8


In [20]:
all_results['config.n_bits']

0      16
1       4
2       8
3       4
4       8
       ..
139     8
140     4
141     4
142     8
143     2
Name: config.n_bits, Length: 144, dtype: int64

## 3. Plot results

In [9]:
all_results["config.model_name"]

0      Llama-3-8B
1      Llama-3-8B
2      Llama-3-8B
3      Llama-3-8B
4      Llama-3-8B
          ...    
139    Llama-3-8B
140    Llama-3-8B
141    Llama-3-8B
142    Llama-3-8B
143    Llama-3-8B
Name: config.model_name, Length: 144, dtype: object

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
from fpdf import FPDF
import os

# Create the plots as before
# Assuming the dataframe `all_results` is already loaded and filtered

# Create the plots
fig1 = px.box(all_results.loc[all_results["config.model_name"] == "Llama-3-8B"], 
              x='config.quantize_method', 
              y='result.perplexity', 
              color='config.n_bits', 
              title='Perplexity for Llama-3-8B',
              labels={'config.quantize_method': 'Quantization Method', 'result.perplexity': 'Perplexity'},
              points='all')
fig1.show()

fig2 = px.box(all_results.loc[all_results["config.model_name"] == "Llama-3-8B"], 
              x='config.quantize_method', 
              y='result.brier_score', 
              color='config.n_bits', 
              title='Brier Score for Llama-3-8B',
              labels={'config.quantize_method': 'Quantization Method', 'result.brier_score': 'Brier Score'},
              points='all')
fig2.show()

fig3 = px.scatter(all_results, 
                  x='config.calib_dataset_name', 
                  y='result.perplexity', 
                  color='config.eval_dataset_name', 
                  symbol='config.eval_dataset_name',
                  title='Perplexity by Calibration and Evaluation Dataset',
                  labels={'config.calib_dataset_name': 'Calibration Dataset', 'result.perplexity': 'Perplexity'})
fig3.show()

fig4 = px.scatter(all_results, 
                  x='config.calib_dataset_name', 
                  y='result.brier_score', 
                  color='config.eval_dataset_name', 
                  symbol='config.eval_dataset_name',
                  title='Brier Score by Calibration and Evaluation Dataset',
                  labels={'config.calib_dataset_name': 'Calibration Dataset', 'result.brier_score': 'Brier Score'})
fig4.show()

fig5 = px.box(all_results[all_results["config.model_name"] == "Llama-3-8B"], 
              x='config.quantize_method', 
              y='result.disk_space_usage', 
              color='config.n_bits', 
              title='Disk Space Usage by Quantization Method and Number of Bits',
              labels={'config.quantize_method': 'Quantization Method', 'result.disk_space_usage': 'Disk Space Usage (GB)'},
              points='all')
fig5.show()

fig6 = px.box(all_results[all_results["config.model_name"] == "Llama-3-8B"], 
              x='config.quantize_method', 
              y='result.quantize_runtime', 
              color='config.n_bits', 
              title='Quantization Runtime by Quantization Method and Number of Bits',
              labels={'config.quantize_method': 'Quantization Method', 'result.quantize_runtime': 'Quantization Runtime (s)'},
              points='all')
fig6.show()

# Visualize the 16 values for the 'AWQ-4' quantization method using a bar plot
awq_4_data = all_results[(all_results['config.quantize_method'] == 'AWQ-4')]

fig7 = px.bar(awq_4_data, 
              x='config.calib_dataset_name', 
              y='result.perplexity', 
              color='config.eval_dataset_name', 
              barmode='group',
              title='Perplexity for AWQ-4 by Calibration and Evaluation Dataset',
              labels={'config.calib_dataset_name': 'Calibration Dataset', 'result.perplexity': 'Perplexity'})
fig7.show()

fig8 = px.bar(awq_4_data, 
              x='config.calib_dataset_name', 
              y='result.brier_score', 
              color='config.eval_dataset_name', 
              barmode='group',
              title='Brier Score for AWQ-4 by Calibration and Evaluation Dataset',
              labels={'config.calib_dataset_name': 'Calibration Dataset', 'result.brier_score': 'Brier Score'})
fig8.show()

In [ ]:
# Create a list of plots and their descriptions
plots = [
    (fig1, "Plot 1: Perplexity by Quantization Method and Number of Bits"),
    (fig2, "Plot 2: Brier Score by Quantization Method and Number of Bits"),
    (fig3, "Plot 3: Perplexity by Calibration and Evaluation Dataset"),
    (fig4, "Plot 4: Brier Score by Calibration and Evaluation Dataset"),
    (fig5, "Plot 5: Disk Space Usage by Quantization Method and Number of Bits"),
    (fig6, "Plot 6: Quantization Runtime by Quantization Method and Number of Bits"),
    (fig7, "Plot 7: Perplexity for AWQ-4 by Calibration and Evaluation Dataset"),
    (fig8, "Plot 8: Brier Score for AWQ-4 by Calibration and Evaluation Dataset")
]

plot_save_path = "plots"
os.makedirs(plot_save_path, exist_ok=True)

# Save plots as images
for i, (fig, desc) in enumerate(plots):
    fig.write_image(os.path.join(plot_save_path, f"plot_{i}.png"))

# Create a PDF document
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)

# Add plots and descriptions to the PDF
for i, (fig, desc) in enumerate(plots):
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, desc)
    pdf.ln(10)
    pdf.image(os.path.join(plot_save_path, f"plot_{i}.png"), w=pdf.w - 30)

# Save the PDF
pdf.output(os.path.join(plot_save_path, "results_summary.pdf"), "F")

print("PDF generated and saved as 'results_summary.pdf'")